# Basic Backtest Tutorial

This notebook walks you through your first AutoPredict backtest.

**Learning objectives:**
- Run a backtest programmatically
- Understand the data format
- Interpret metrics
- Visualize results

**Time**: 15 minutes

## Setup

In [ ]:
import json
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from agent import AutoPredictAgent, AgentConfig, MarketState
from market_env import OrderBook, BookLevel, ExecutionEngine, evaluate_all
from run_experiment import run_backtest

## Step 1: Load Market Data

Market data is stored as JSON with this format:
- `market_id`: Unique identifier
- `market_prob`: Current market price (0-1)
- `fair_prob`: Your forecast (0-1)
- `outcome`: Actual result (0 or 1)
- `order_book`: Bids and asks with depth

In [ ]:
# Load sample markets
dataset_path = Path.cwd().parent / "datasets" / "sample_markets.json"

with open(dataset_path) as f:
    markets = json.load(f)

print(f"Loaded {len(markets)} markets")
print("\nFirst market:")
print(json.dumps(markets[0], indent=2))

## Step 2: Explore the Data

In [ ]:
# Calculate edges
edges = [m["fair_prob"] - m["market_prob"] for m in markets]

print("Edge statistics:")
print(f"  Min: {min(edges):.3f}")
print(f"  Max: {max(edges):.3f}")
print(f"  Mean: {sum(edges) / len(edges):.3f}")
print(f"  Positive edges: {sum(1 for e in edges if e > 0)}")
print(f"  Negative edges: {sum(1 for e in edges if e < 0)}")

In [ ]:
# Visualize edge distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(edges, bins=20, edgecolor='black')
plt.axvline(0, color='red', linestyle='--', label='Zero edge')
plt.xlabel('Edge (fair_prob - market_prob)')
plt.ylabel('Frequency')
plt.title('Distribution of Edges')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Step 3: Configure Agent Strategy

The agent config controls all decision-making parameters.

In [ ]:
# Create baseline config
config = AgentConfig(
    min_edge=0.05,              # Minimum edge to trade
    aggressive_edge=0.12,       # Threshold for market orders
    max_risk_fraction=0.02,     # Max 2% risk per trade
    max_position_notional=25.0, # Max $25 per trade
    min_book_liquidity=60.0,    # Min total depth
    max_spread_pct=0.04,        # Max 4% spread
    max_depth_fraction=0.15     # Max 15% of visible depth
)

print("Agent config:")
print(f"  min_edge: {config.min_edge}")
print(f"  aggressive_edge: {config.aggressive_edge}")
print(f"  max_risk_fraction: {config.max_risk_fraction}")

## Step 4: Run Backtest

The backtest simulates trading on all markets.

In [ ]:
# Initialize agent
agent = AutoPredictAgent(config)
bankroll = 1000.0

# Run backtest manually (for visibility)
trades = []
forecasts = []
outcomes = []

for market_data in markets:
    # Convert to MarketState
    market = MarketState(
        market_id=market_data["market_id"],
        market_prob=market_data["market_prob"],
        fair_prob=market_data["fair_prob"],
        time_to_expiry_hours=market_data.get("time_to_expiry_hours", 24.0),
        order_book=OrderBook(
            market_id=market_data["market_id"],
            bids=[BookLevel(p, s) for p, s in market_data["order_book"]["bids"]],
            asks=[BookLevel(p, s) for p, s in market_data["order_book"]["asks"]]
        )
    )
    
    # Evaluate market
    order = agent.evaluate_market(market, bankroll)
    
    # Record forecast
    forecasts.append(market.fair_prob)
    outcomes.append(market_data["outcome"])
    
    if order:
        print(f"Trade: {order.side} {order.size:.2f} @ {market.market_prob:.3f} (edge={market.fair_prob - market.market_prob:+.3f})")
        
        # Simulate execution
        engine = ExecutionEngine()
        if order.order_type == "market":
            result = engine.execute_market_order(
                market.order_book,
                order.side,
                order.size
            )
        else:
            result = engine.execute_limit_order(
                market.order_book,
                order.side,
                order.size,
                order.limit_price
            )
        
        # Record trade
        trades.append({
            "market_id": market.market_id,
            "side": order.side,
            "order_type": order.order_type,
            "size": order.size,
            "filled_size": result["filled_size"],
            "fill_price": result["avg_fill_price"],
            "slippage_bps": result["slippage_bps"],
            "outcome": market_data["outcome"],
            "pnl": result["filled_size"] * (market_data["outcome"] - result["avg_fill_price"]) if order.side == "buy" else result["filled_size"] * (result["avg_fill_price"] - market_data["outcome"])
        })

print(f"\nTotal trades: {len(trades)}")

## Step 5: Calculate Metrics

In [ ]:
# Calculate Brier score
brier = sum((f - o) ** 2 for f, o in zip(forecasts, outcomes)) / len(forecasts)

# Calculate financial metrics
total_pnl = sum(t["pnl"] for t in trades)
pnls = [t["pnl"] for t in trades]
mean_pnl = sum(pnls) / len(pnls) if pnls else 0
std_pnl = (sum((p - mean_pnl) ** 2 for p in pnls) / len(pnls)) ** 0.5 if len(pnls) > 1 else 0
sharpe = mean_pnl / std_pnl * (len(pnls) ** 0.5) if std_pnl > 0 else 0

# Calculate execution metrics
avg_slippage = sum(t["slippage_bps"] for t in trades) / len(trades) if trades else 0
fill_rate = sum(t["filled_size"] for t in trades) / sum(t["size"] for t in trades) if trades else 0

print("Backtest Results:")
print("\nFinancial:")
print(f"  Total PnL: ${total_pnl:.2f}")
print(f"  Sharpe: {sharpe:.2f}")
print(f"  Mean PnL per trade: ${mean_pnl:.2f}")
print("\nEpistemic:")
print(f"  Brier score: {brier:.3f}")
print("\nExecution:")
print(f"  Avg slippage: {avg_slippage:.1f} bps")
print(f"  Fill rate: {fill_rate:.1%}")
print(f"  Trades executed: {len(trades)}")

## Step 6: Visualize Performance

In [ ]:
# Cumulative PnL
cumulative_pnl = []
running_total = 0
for trade in trades:
    running_total += trade["pnl"]
    cumulative_pnl.append(running_total)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(cumulative_pnl, linewidth=2)
plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.xlabel('Trade Number')
plt.ylabel('Cumulative PnL ($)')
plt.title('Cumulative PnL Over Time')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist([t["pnl"] for t in trades], bins=15, edgecolor='black')
plt.axvline(0, color='red', linestyle='--', label='Break-even')
plt.xlabel('Trade PnL ($)')
plt.ylabel('Frequency')
plt.title('Distribution of Trade PnL')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 7: Experiment with Different Parameters

Try changing `min_edge` to see how it affects results.

In [ ]:
# Test different min_edge values
min_edges = [0.03, 0.05, 0.08, 0.10, 0.12]
results = []

for min_edge in min_edges:
    config = AgentConfig(min_edge=min_edge)
    agent = AutoPredictAgent(config)
    
    # Simplified backtest
    trades_count = 0
    for market_data in markets:
        market = MarketState(
            market_id=market_data["market_id"],
            market_prob=market_data["market_prob"],
            fair_prob=market_data["fair_prob"],
            time_to_expiry_hours=24.0,
            order_book=OrderBook(
                market_id=market_data["market_id"],
                bids=[BookLevel(p, s) for p, s in market_data["order_book"]["bids"]],
                asks=[BookLevel(p, s) for p, s in market_data["order_book"]["asks"]]
            )
        )
        if agent.evaluate_market(market, 1000.0):
            trades_count += 1
    
    results.append(trades_count)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(min_edges, results, marker='o', linewidth=2, markersize=8)
plt.xlabel('min_edge')
plt.ylabel('Number of Trades')
plt.title('Trade Count vs min_edge Threshold')
plt.grid(alpha=0.3)
plt.show()

print("Trade counts by min_edge:")
for edge, count in zip(min_edges, results):
    print(f"  {edge:.2f}: {count} trades")

## Next Steps

- Try notebook `02_strategy_comparison.ipynb` to compare multiple strategies
- Read `STRATEGIES.md` to learn how to build custom strategies
- Read `BACKTESTING.md` for rigorous validation techniques